In [2]:
# Step 1: Import libraries
import pandas as pd
import os
import time
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter

# Step 2: Load dataset
df = pd.read_csv("place2001-2010.csv")

# Step 3: Clean column names
df.columns = df.columns.str.strip()

df["State_Name"] = df["State_Name"].astype(str).str.strip()
df["District_Name"] = df["District_Name"].astype(str).str.strip()

# Remove invalid rows
df = df.dropna(subset=["State_Name", "District_Name"])
df = df[
    (df["State_Name"] != "") &
    (df["District_Name"] != "") &
    (df["State_Name"].str.lower() != "nan") &
    (df["District_Name"].str.lower() != "nan")
].reset_index(drop=True)

# Step 4: Initialize geocoder
geolocator = Nominatim(
    user_agent="agriculture_weather_project_2026"
)

geocode = RateLimiter(
    geolocator.geocode,
    min_delay_seconds=1,
    max_retries=3,
    error_wait_seconds=5,
    swallow_exceptions=True
)

# Step 5: Geocode unique state-district combinations
cache = {}
results = []

# Reuse existing coordinates if available
if os.path.exists("coordinates_all.csv"):
    old = pd.read_csv("coordinates_all.csv")

    for _, row in old.iterrows():
        if pd.notna(row.get("latitude")) and pd.notna(row.get("longitude")):
            key = (
                str(row["State_Name"]).strip().casefold(),
                str(row["District_Name"]).strip().casefold()
            )
            cache[key] = (row["latitude"], row["longitude"])

for i, row in df.iterrows():
    state = row["State_Name"]
    district = row["District_Name"]

    key = (state.casefold(), district.casefold())

    if key in cache:
        lat, lon = cache[key]
    else:
        query = f"{district}, {state}, India"

        try:
            location = geocode(query)

            if location:
                lat = location.latitude
                lon = location.longitude
                cache[key] = (lat, lon)
            else:
                lat, lon = None, None

        except Exception as e:
            print(f"Error for {query}: {e}")
            lat, lon = None, None

    results.append({
        "State_Name": state,
        "District_Name": district,
        "latitude": lat,
        "longitude": lon
    })

    # Save progress after every 10 rows
    if (i + 1) % 10 == 0:
        pd.DataFrame(results).to_csv(
            "coordinates_progress.csv",
            index=False
        )
        print(f"Processed {i + 1}/{len(df)} rows")

# Step 6: Save all coordinate records
coordinates = pd.DataFrame(results)

coordinates.to_csv("coordinates_all.csv", index=False)

# Step 7: Create first 300 records
coordinates.iloc[:300].to_csv(
    "coor_300.csv",
    index=False
)

# Step 8: Create next 300 records (301–600)
coordinates.iloc[300:600].to_csv(
    "coor_600.csv",
    index=False
)

# Step 9: Display output
print("\nAll files created successfully!")
print("Total records:", len(coordinates))
print("coor_300.csv records:", len(coordinates.iloc[:300]))
print("coor_600.csv records:", len(coordinates.iloc[300:600]))

print("\nMissing coordinates:")
print(coordinates[["latitude", "longitude"]].isnull().sum())

print("\nFiles saved in:")
print(os.getcwd())

print("\nPreview:")
print(coordinates.head())

Processed 10/621 rows
Processed 20/621 rows
Processed 30/621 rows
Processed 40/621 rows
Processed 50/621 rows
Processed 60/621 rows
Processed 70/621 rows
Processed 80/621 rows
Processed 90/621 rows
Processed 100/621 rows
Processed 110/621 rows
Processed 120/621 rows
Processed 130/621 rows
Processed 140/621 rows
Processed 150/621 rows
Processed 160/621 rows
Processed 170/621 rows
Processed 180/621 rows
Processed 190/621 rows
Processed 200/621 rows
Processed 210/621 rows
Processed 220/621 rows
Processed 230/621 rows
Processed 240/621 rows
Processed 250/621 rows
Processed 260/621 rows
Processed 270/621 rows
Processed 280/621 rows
Processed 290/621 rows
Processed 300/621 rows
Processed 310/621 rows
Processed 320/621 rows
Processed 330/621 rows
Processed 340/621 rows
Processed 350/621 rows
Processed 360/621 rows
Processed 370/621 rows
Processed 380/621 rows
Processed 390/621 rows
Processed 400/621 rows
Processed 410/621 rows
Processed 420/621 rows
Processed 430/621 rows
Processed 440/621 ro